In [ ]:
import sys
sys.executable

!"{sys.executable}" -m pip install numpy pandas matplotlib statadict empiricaldist statsmodels

from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)

download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")   # downloads thinkstats module

try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels
from IPython.display import HTML
from thinkstats import decorate     # think stats is not a pip package


# note the code below can be a generic way to install package
try:
   import statadict
except ImportError:
    %pip install statadict
    import statadict


In [6]:
# The data is stored in two files (download), a “dictionary” that describes the format of the data, and a data file:
import os
print(os.getcwd())

from statadict import parse_stata_dict

 # paste path name under "...", if Mac dont forget to remove ''
dat_file = "/Users/saadmanchowdhury/Desktop/All Github projects/01._eda_and_regression/data/2002FemPreg.dat.gz"   # A dat file is a type of data file that is used to hold settings and information for a certain piece of software like  plain text, video, audio, and even PDFs. dat files don’t contain column names or structure, just raw text
dct_file = "/Users/saadmanchowdhury/Desktop/All Github projects/01._eda_and_regression/data/2002FemPreg.dct"  # A Stata .dct file is a data dictionary file used to read raw, fixed-format ASCII data (often .dat or .txt) into Stata. The .dct file tells you:Column names,Column positions,Column widths


In [ ]:
# reading data for Stata

from statadict import parse_stata_dict


# help(parse_stata_dict)   # this is very useful if stuck, Parses Stata dictionary file and returns object containing column data as attributes.
# help(pd.read_fwf)        # A comma-separated values (csv) file is returned as two-dimensional data structure with labeled axes.

# names is a general argument supported by most pandas file readers (read_csv, read_fwf, etc.) It sets the column names of the DataFrame.

def read_stata(dct_file, dat_file):
    stata_dict = parse_stata_dict(dct_file)
    resp = pd.read_fwf(
        dat_file,
        names=stata_dict.names,
        colspecs=stata_dict.colspecs,
        compression="gzip",
    )
    return resp

# read_fwf accepts many extra parameters that are documented in the full pandas IO docs.










In [ ]:
# Exploratory Data analysis Start

# load data
preg = read_stata(dct_file, dat_file)  

#  EDA start
# Show dimensions with .shape which come from numpy
preg.shape




NameError: name 'read_stata' is not defined

In [11]:
preg.columns


In [13]:
pregordr = preg["pregordr"]

type(pregordr)                  # The result is a Pandas Series, which represents a sequence of values. 

In [14]:
pregordr.head()

In [ ]:
# Data Validation

preg["outcome"].value_counts().sort_index()   # this code is intersting becuase it counts how many for each outcome, sort_index, which sorts the results according to the values in the Index (left column)
counts = preg["birthwgt_lb"].value_counts(dropna=False).sort_index()
counts

In [ ]:
counts.loc[0:5]    # note this is from pandas

NameError: name 'pandas' is not defined

In [24]:
counts.loc[0:5].sum()   # sums the counts


In [ ]:
# Data Cleaning

preg["birthwgt_lb"] = preg["birthwgt_lb"].replace([51, 97, 98, 99], np.nan)  #replace data for values 51,97,98,99 with NaN ie missing data

In [26]:
preg["agepreg"].mean()
preg["agepreg"] /= 100.0    # note this operation means variable/100 = variable
preg["agepreg"].mean()

In [ ]:
preg["birthwgt_oz"] = preg["birthwgt_oz"].replace([97, 98, 99], np.nan)

preg["totalwgt_lb"] = preg["birthwgt_lb"] + preg["birthwgt_oz"] / 16.0    # 1 pound is 16 oz, Birth weight in many medical and survey datasets is recorded like this: X pounds and Y ounces

preg["totalwgt_lb"].mean()

In [32]:
#Summary statistics

weights = preg["totalwgt_lb"]
n = weights.count()            # counts the number of non-missing values only, not the value counts
n                              # therefore we have 9039 valid rows not 13539!


In [ ]:
mean = weights.sum() / n
mean                            # average weight

weights.mean()                  #same thing

In [ ]:
squared_deviations = (weights - mean) ** 2   # code to calculate squared deviations vector

var = squared_deviations.sum() / n
var

weights.var()      # just use this code to calculate variance, note this uses n-1 so is more accurate

In [38]:
std = np.sqrt(var)   # numpy sqrt function
std

weights.std(ddof=0)

In [ ]:
subset = preg.query("caseid == 10229")     # this is a great example of pulling data like SQL, returns only rows where caseid == 10229
subset.shape

In [ ]:
subset["outcome"].values    # creating a vector of values from a particular column fro a dataset